# UC-Cap: a hybrid statistical-learning risk score

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FEI-JIANG/uccap/blob/main/examples/notebooks/03_uccap_hybrid_risk.ipynb)

Companion notebook to F. Jiang and L. Yang, *A hybrid statistical learning framework for
capability-based decision support in manufacturing*, Int. J. Adv. Manuf. Technol. (2026),
[doi:10.1007/s00170-026-18833-9](https://doi.org/10.1007/s00170-026-18833-9)
([arXiv:2604.13352](https://arxiv.org/abs/2604.13352)).

UC-Cap estimates the probability that the true Ppk is below 1.33 as

π = σ( z + f(x) ),  z = (1.33 − Ppk) / SE(Ppk),

a statistical anchor z plus a correction f learned by a small neural network from the shape of the
data and its position in the tolerance. The trained parameters of the network (V4.2) are not
included in this release; they are available from the authors on request. This notebook shows the
statistical part, which works without them, and runs the model cells only when the parameters are
found.

In [1]:
# On Colab or another fresh environment, install uccap first
try:
    import uccap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "uccap[plot] @ git+https://github.com/FEI-JIANG/uccap"], check=True)

In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import uccap
np.set_printoptions(precision=4, suppress=True)

## 0. Model parameters

Give the location of the parameter file with `uccap.hybrid.set_model_path(path)` or the environment
variable `UCCAP_MODEL`.

In [3]:
import uccap.hybrid
# uccap.hybrid.set_model_path("path/to/uccap_v4_2.json")
HAVE_MODEL = uccap.hybrid.model_available()
print("UC-Cap parameters found" if HAVE_MODEL else "UC-Cap parameters not found: the model cells are skipped")

UC-Cap parameters not found: the model cells are skipped


## 1. One dimension

The statistical anchor and the mechanism signals of the decision layer need no model parameters.

In [4]:
from uccap.hybrid import compute_features, mechanism

x = np.random.default_rng(11).normal(10.012, 0.011, 20)    # 20 parts, tolerance 10.00 ± 0.05
f = compute_features(x, 9.95, 10.05, 10.0)
z = f["A_z_stat_133"]
print(f"Ppk {f['A_Ppk']:.4f}   SE(Ppk) {f['A_SE_Ppk']:.4f}   anchor z {z:.4f}   anchor only {1 / (1 + np.exp(-z)):.4f}")
print("mechanism signals:", mechanism.signals(f).active)

Ppk 1.3654   SE(Ppk) 0.2828   anchor z -0.1251   anchor only 0.4688
mechanism signals: ('shift',)


The anchor alone, σ(z), is the purely statistical risk. Here it is close to 0.5, so the statistics
alone are undecided. The model inputs are available for inspection.

In [5]:
{k: round(v, 4) if isinstance(v, float) else v for k, v in f.items()}

{'nominal': 10.0,
 'USL': 10.05,
 'LSL': 9.95,
 'Spec_Width': 0.1,
 'A_Ppk': 1.3654,
 'A_Pp': 1.9055,
 'A_mean': 10.0142,
 'A_sd': 0.0087,
 'A_Skewness': -0.4142,
 'A_Kurtosis': 3.4754,
 'A_Dist_to_LSL': 0.0642,
 'A_Dist_to_USL': 0.0358,
 'A_Dist_to_nearest_spec': 0.0358,
 'A_P_value': 0.2886,
 'A_SE_Ppk': 0.2828,
 'A_margin_to_threshold': 0.0354,
 'A_normality': 'Pass',
 'A_Best_Fit': 'normal',
 'A_z_stat_133': -0.1251,
 'Fit_Pp': nan,
 'Fit_Ppk': nan,
 'Upper_only': False}

With the parameters, the learned correction moves the anchor on the logit scale and the decision layer
adds a level, a reason and an action.

In [6]:
if HAVE_MODEL:
    r = uccap.hybrid_risk(x, lsl=9.95, usl=10.05, nominal=10.0)
    print(r)
else:
    print("skipped: the trained UC-Cap parameters are not included in this release")

skipped: the trained UC-Cap parameters are not included in this release


## 2. Risk levels

The score is 100 π. Levels (hybrid article, Eq. 26): **Low** below τL = 0.40 (accept or continue
routine monitoring), **Medium** 0.40–0.70 (review or collect additional evidence), **High** from
τH = 0.70 (reject, hold or initiate corrective action). The thresholds are prespecified and can be
replaced; the engineer confirms the final action. Below, the same tolerance with processes of
decreasing capability.

In [7]:
for sd in (0.008, 0.010, 0.012, 0.014, 0.016, 0.020):
    xs = np.random.default_rng(5).normal(10.0, sd, 20)
    fs = compute_features(xs, 9.95, 10.05)
    line = f"sd {sd:.3f}   Ppk {fs['A_Ppk']:5.2f}   anchor {1 / (1 + np.exp(-fs['A_z_stat_133'])):.3f}"
    if HAVE_MODEL:
        h = uccap.hybrid_risk(xs, 9.95, 10.05)
        line += f"   UC-Cap {h.p:.3f}   {h.level}"
    print(line)

sd 0.008   Ppk  2.11   anchor 0.092
sd 0.010   Ppk  1.67   anchor 0.215
sd 0.012   Ppk  1.38   anchor 0.440
sd 0.014   Ppk  1.17   anchor 0.703
sd 0.016   Ppk  1.02   anchor 0.882
sd 0.020   Ppk  0.80   anchor 0.988


## 3. Next to the risk-calibrated decision

UC-Cap and the risk-calibrated rule of the companion article answer related questions. With the model
parameters, `batch_study` reports both for every dimension with two limits or an upper limit only:
`decision` and `p_fail` from the rule, `risk_score` and `risk_level` from UC-Cap.

In [8]:
import pandas as pd
rng = np.random.default_rng(8)
data = pd.DataFrame({f"D{i}": rng.normal(10 + shift, sd, 20)
                     for i, (shift, sd) in enumerate([(0, .008), (.01, .010), (0, .013), (.015, .011), (0, .018)], 1)})
specs = {d: {"lsl": 9.95, "usl": 10.05, "target": 10.0} for d in data}
res = uccap.batch_study(data, specs, c0=1.33, cost_ratio=19, hybrid=HAVE_MODEL)
cols = ["dimension", "Ppk", "p_fail", "decision"] + (["risk_score", "risk_level"] if HAVE_MODEL else [])
res.table(cols).round(3)

,dimension,Ppk,p_fail,decision
0,D1,1.834,0.046,ACCEPT
1,D2,1.146,0.827,REJECT
2,D3,1.110,0.876,REJECT
3,D4,0.997,0.972,REJECT
4,D5,0.931,0.992,REJECT


## 4. Scope

The network was trained on real dimensional data with about 20 measurements per characteristic,
bilateral tolerances or an upper limit only (coded with LSL = 0), and C0 = 1.33. `hybrid_risk` therefore
needs both limits or an upper limit, and at least 8 values; for other sample sizes or requirements treat
the score as indicative and rely on the risk-calibrated decision. The training data are not public; the
article reports the model's test metrics (Table 10), which the model parameters reproduced when they
were exported.